# Parkinson's Disease Voice Screening Platform
## 10-Second End-to-End Training & Export Pipeline (Google Colab Free T4 GPU)

> **CLINICAL RESEARCH NOTICE:** This platform is a clinical decision support and risk stratification research tool, **NOT** an autonomous diagnostic device. It does not replace comprehensive neurological evaluation, UPDRS scoring, or clinical examination.

---

### How to Run in 3 Steps:
1. **Enable T4 GPU:** Go to **Runtime** > **Change runtime type** > Select **T4 GPU** > Click **Save**.
2. **Run All Cells:** Click **Runtime** > **Run all** (or press `Ctrl+F9` / `Cmd+F9`).
3. **Download Weights:** The final cell will automatically download `models_artifact_10s.zip` to your computer!

*Total execution time:* **~4 to 5 minutes** on free Colab T4 GPU.

In [ ]:
# Cell 1: Environment Setup, Dependencies & Hardware Verification
import sys
import os
import subprocess
from pathlib import Path

# Prevent CUDA memory fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("Running in Google Colab environment.")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "transformers", "torch", "torchaudio", "soundfile", "librosa", "pandas", "numpy", "tqdm", "scikit-learn", "huggingface_hub", "requests", "matplotlib"
    ])
    from google.colab import drive, files

    REPO_DIR = Path("/content/prototype")
    if not REPO_DIR.exists():
        print("Cloning repository into Colab...")
        subprocess.check_call(["git", "clone", "https://github.com/eshwarsaielugam11-max/prototype.git", str(REPO_DIR)])
    
    PROJECT_ROOT = REPO_DIR
else:
    print("Running in local development environment.")
    cwd = Path.cwd()
    PROJECT_ROOT = cwd.parent.parent if cwd.name == "notebooks" else cwd

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project Root: {PROJECT_ROOT}")

import torch
print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"CUDA Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM Available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Apple Silicon MPS Active.")
else:
    device = torch.device("cpu")
    print("Using CPU Device.")

In [ ]:
# Cell 2: Automated Acquisition of Datasets (IPVS from Hugging Face & MDVR-KCL from Zenodo)
import os
import json
import zipfile
import urllib.request
import pandas as pd
from huggingface_hub import snapshot_download

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

IPVS_DIR = RAW_DIR / "ipvs"
MDVR_DIR = RAW_DIR / "mdvr_kcl"
AUDIO_10S_DIR = PROCESSED_DIR / "audio_10s"
FEATURES_10S_DIR = PROCESSED_DIR / "features_10s"

for d in [RAW_DIR, PROCESSED_DIR, AUDIO_10S_DIR, FEATURES_10S_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# 1. Download / Verify IPVS Dataset
if not IPVS_DIR.exists() or len(list(IPVS_DIR.rglob("*.wav"))) == 0:
    print("Downloading IPVS dataset from Hugging Face mirror (birgermoell/Italian_Parkinsons_Voice_and_Speech)...")
    IPVS_DIR.mkdir(parents=True, exist_ok=True)
    snapshot_download(
        repo_id="birgermoell/Italian_Parkinsons_Voice_and_Speech",
        repo_type="dataset",
        local_dir=str(IPVS_DIR),
        local_dir_use_symlinks=False,
        resume_download=True
    )

ipvs_wav_count = len(list(IPVS_DIR.rglob("*.wav")))
print(f"✓ IPVS Dataset verified: {ipvs_wav_count} audio files in {IPVS_DIR}")

# 2. Download / Verify MDVR-KCL Dataset from Zenodo
MDVR_READ_TEXT_DIR = MDVR_DIR / "extracted" / "26-29_09_2017_KCL" / "ReadText"
if not MDVR_READ_TEXT_DIR.exists():
    alt_dir = MDVR_DIR / "ReadText"
    if alt_dir.exists():
        MDVR_READ_TEXT_DIR = alt_dir

if not MDVR_READ_TEXT_DIR.exists() or len(list(MDVR_READ_TEXT_DIR.glob("*/*.wav"))) == 0:
    print("Downloading MDVR-KCL dataset archive from Zenodo (Record 2867216)...")
    MDVR_DIR.mkdir(parents=True, exist_ok=True)
    ZENODO_URL = "https://zenodo.org/records/2867216/files/26_29_09_2017_KCL.zip?download=1"
    zip_dest = MDVR_DIR / "26_29_09_2017_KCL.zip"
    if not zip_dest.exists():
        print(f"Downloading {ZENODO_URL} -> {zip_dest}...")
        urllib.request.urlretrieve(ZENODO_URL, str(zip_dest))
        print(f"Downloaded {zip_dest.name} ({zip_dest.stat().st_size / 1e6:.1f} MB)")
    
    print("Extracting MDVR-KCL archive...")
    extract_target = MDVR_DIR / "extracted"
    extract_target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(str(zip_dest), "r") as zf:
        zf.extractall(str(extract_target))
    MDVR_READ_TEXT_DIR = extract_target / "26-29_09_2017_KCL" / "ReadText"

assert MDVR_READ_TEXT_DIR.exists(), f"MDVR ReadText directory missing at {MDVR_READ_TEXT_DIR}"
mdvr_wav_count = len(list(MDVR_READ_TEXT_DIR.glob("*/*.wav")))
print(f"✓ MDVR-KCL ReadText verified: {mdvr_wav_count} audio files across HC and PD cohorts.")

In [ ]:
# Cell 3: Subject-Level Stratification & Strict Leakage Audit
IPVS_MANIFEST_PATH = PROCESSED_DIR / "split_manifest.json"
assert IPVS_MANIFEST_PATH.exists(), f"Split manifest missing at {IPVS_MANIFEST_PATH}"

with open(IPVS_MANIFEST_PATH, "r", encoding="utf-8") as f:
    ipvs_manifest = json.load(f)

ipvs_split_map = {}
for split_name in ["train", "val", "test"]:
    for sid in ipvs_manifest[split_name]:
        ipvs_split_map[sid] = split_name

MDVR_SPLITS = {
    "train": [
        "ID00", "ID28", "ID25", "ID01", "ID12", "ID09", "ID19", "ID05", "ID31", "ID26", "ID22", "ID03", "ID14", "ID36", "ID08",
        "ID18", "ID32", "ID27", "ID07", "ID34", "ID17", "ID06", "ID30", "ID20", "ID16", "ID24"
    ],
    "val": ["ID21", "ID11", "ID15", "ID33", "ID02"],
    "test": ["ID23", "ID35", "ID10", "ID13", "ID29", "ID04"]
}

mdvr_split_map = {}
for split_name, sids in MDVR_SPLITS.items():
    for sid in sids:
        mdvr_split_map[sid] = split_name

train_subjects = set(ipvs_manifest["train"]) | set(MDVR_SPLITS["train"])
val_subjects = set(ipvs_manifest["val"]) | set(MDVR_SPLITS["val"])
test_subjects = set(ipvs_manifest["test"]) | set(MDVR_SPLITS["test"])

print("=" * 60)
print("SUBJECT-LEVEL STRATIFICATION AUDIT")
print("=" * 60)
print(f"  Train Subjects:       {len(train_subjects):3d} (IPVS: {len(ipvs_manifest['train'])}, MDVR: {len(MDVR_SPLITS['train'])})")
print(f"  Validation Subjects:  {len(val_subjects):3d} (IPVS: {len(ipvs_manifest['val'])}, MDVR: {len(MDVR_SPLITS['val'])})")
print(f"  Test Subjects:        {len(test_subjects):3d} (IPVS: {len(ipvs_manifest['test'])}, MDVR: {len(MDVR_SPLITS['test'])})")
print(f"  Total Unique Subjects:{len(train_subjects | val_subjects | test_subjects):3d}")

# Strict zero leakage assertions
assert len(train_subjects & val_subjects) == 0, f"Leakage between Train & Val: {train_subjects & val_subjects}"
assert len(train_subjects & test_subjects) == 0, f"Leakage between Train & Test: {train_subjects & test_subjects}"
assert len(val_subjects & test_subjects) == 0, f"Leakage between Val & Test: {val_subjects & test_subjects}"

print("\n✓ PASS: Zero subject leakage confirmed across all splits.")

In [ ]:
# Cell 4: 10-Second Window Extraction Pipeline
import soundfile as sf
import numpy as np
import librosa
from dataclasses import dataclass
from typing import List, Tuple
from tqdm import tqdm

@dataclass
class WindowConfig:
    target_sr: int = 16000
    trim_top_db: int = 30
    normalize: bool = True
    segment_seconds: float = 10.0
    window_stride: float = 5.0
    min_valid_seconds: float = 8.0
    max_pad_fraction: float = 0.20
    pad_mode: str = "repeat"

    @property
    def target_samples(self) -> int:
        return int(round(self.segment_seconds * self.target_sr))

def extract_windows_from_waveform(waveform: np.ndarray, sample_rate: int, config: WindowConfig) -> List[Tuple[np.ndarray, float, float]]:
    if waveform is None or waveform.size == 0:
        return []

    if np.issubdtype(waveform.dtype, np.integer):
        max_val = float(np.iinfo(waveform.dtype).max)
        waveform = waveform.astype(np.float32) / max_val
    else:
        waveform = waveform.astype(np.float32)

    if waveform.ndim > 1:
        waveform = np.mean(waveform, axis=0) if waveform.shape[0] < waveform.shape[1] else np.mean(waveform, axis=-1)
    waveform = waveform.flatten()

    if sample_rate != config.target_sr:
        waveform = librosa.resample(waveform, orig_sr=sample_rate, target_sr=config.target_sr, res_type="soxr_hq")

    trimmed, _ = librosa.effects.trim(waveform, top_db=config.trim_top_db)
    if len(trimmed) == 0:
        return []

    if config.normalize:
        peak = np.max(np.abs(trimmed))
        if peak > 1e-8:
            trimmed = trimmed / peak

    trimmed_dur = len(trimmed) / config.target_sr
    target_samples = config.target_samples

    if trimmed_dur < config.min_valid_seconds:
        return []

    if trimmed_dur < config.segment_seconds:
        pad_fraction = float((config.segment_seconds - trimmed_dur) / config.segment_seconds)
        repeats = int(np.ceil(target_samples / max(len(trimmed), 1)))
        window = np.tile(trimmed, repeats)[:target_samples]
        return [(np.ascontiguousarray(window, dtype=np.float32), 0.0, pad_fraction)]

    stride_samples = int(round(config.window_stride * config.target_sr))
    windows = []
    start_sample = 0
    while start_sample + target_samples <= len(trimmed):
        win = trimmed[start_sample : start_sample + target_samples]
        start_sec = float(start_sample / config.target_sr)
        windows.append((np.ascontiguousarray(win, dtype=np.float32), start_sec, 0.0))
        start_sample += stride_samples

    return windows

win_config = WindowConfig()
print("WindowConfig initialized: 10.0s windows, 5.0s stride, 8.0s minimum, repeat-padding <= 20%.")

In [ ]:
# Cell 5: Process Audio Files and Generate metadata_10s.csv
ipvs_records = []
for wav_path in sorted(list(IPVS_DIR.rglob("*.wav"))):
    parts = wav_path.parts
    try:
        group_idx = next(i for i, p in enumerate(parts) if any(k in p for k in ["Young", "Elderly", "Parkinson"]))
    except StopIteration:
        continue
    group_str = parts[group_idx]
    fname = wav_path.name
    if "Young" in group_str:
        label = 0
        sub_name = parts[group_idx + 1].strip().replace(" ", "_")
        subject_id = f"yhc_{sub_name}"
    elif "Elderly" in group_str:
        label = 0
        sub_name = parts[group_idx + 1].strip().replace(" ", "_")
        subject_id = f"ehc_{sub_name}"
    elif "Parkinson" in group_str:
        label = 1
        batch = parts[group_idx + 1].strip().replace(" ", "_")
        sub_name = parts[group_idx + 2].strip().replace(" ", "_")
        subject_id = f"pd_{batch}_{sub_name}"
    else:
        continue

    fname_upper = fname.upper()
    if fname_upper.startswith(("VA", "VE", "VI", "VO", "VU")):
        task_type = "vowel_sustain"
    elif fname_upper.startswith("PR"):
        task_type = "reading_passage"
    elif fname_upper.startswith(("B1", "B2", "D1", "D2", "FB1")):
        task_type = "syllable_repetition"
    else:
        task_type = "syllable_repetition"

    if subject_id in ipvs_split_map:
        ipvs_records.append({
            "source_path": wav_path,
            "subject_id": subject_id,
            "dataset": "ipvs",
            "task_type": task_type,
            "label": label,
            "split": ipvs_split_map[subject_id],
            "file_stem": wav_path.stem
        })

mdvr_records = []
for cls_folder, label in [("HC", 0), ("PD", 1)]:
    cls_dir = MDVR_READ_TEXT_DIR / cls_folder
    if not cls_dir.exists():
        continue
    for wav_path in sorted(list(cls_dir.glob("*.wav"))):
        sid = wav_path.stem.split("_")[0]
        if sid in mdvr_split_map:
            mdvr_records.append({
                "source_path": wav_path,
                "subject_id": sid,
                "dataset": "mdvr_kcl",
                "task_type": "reading_passage",
                "label": label,
                "split": mdvr_split_map[sid],
                "file_stem": wav_path.stem
            })

all_recordings = ipvs_records + mdvr_records
print(f"Total recordings discovered: {len(all_recordings)} (IPVS: {len(ipvs_records)}, MDVR: {len(mdvr_records)})")

metadata_rows = []
for rec in tqdm(all_recordings, desc="Extracting 10s Windows"):
    try:
        audio, sr = sf.read(str(rec["source_path"]))
    except Exception as exc:
        continue

    windows = extract_windows_from_waveform(audio, sr, win_config)
    if not windows:
        continue

    subj_audio_dir = AUDIO_10S_DIR / rec["split"] / rec["subject_id"]
    subj_feat_dir = FEATURES_10S_DIR / rec["split"] / rec["subject_id"]
    subj_audio_dir.mkdir(parents=True, exist_ok=True)
    subj_feat_dir.mkdir(parents=True, exist_ok=True)

    for idx, (win_arr, start_sec, pad_frac) in enumerate(windows):
        win_fname = f"{rec['file_stem']}_w{idx:03d}_{start_sec:.1f}s.wav"
        feat_fname = f"{rec['file_stem']}_w{idx:03d}_{start_sec:.1f}s.npy"

        win_path = subj_audio_dir / win_fname
        feat_path = subj_feat_dir / feat_fname

        if not win_path.exists():
            sf.write(str(win_path), win_arr, win_config.target_sr, subtype="PCM_16")

        metadata_rows.append({
            "window_path": win_path.relative_to(PROJECT_ROOT).as_posix(),
            "feature_path": feat_path.relative_to(PROJECT_ROOT).as_posix(),
            "subject_id": rec["subject_id"],
            "dataset": rec["dataset"],
            "task_type": rec["task_type"],
            "label": rec["label"],
            "split": rec["split"],
            "window_start_sec": round(float(start_sec), 2),
            "padded_fraction": round(float(pad_frac), 4)
        })

df_meta = pd.DataFrame(metadata_rows)
METADATA_10S_PATH = PROCESSED_DIR / "metadata_10s.csv"
df_meta.to_csv(METADATA_10S_PATH, index=False)
print(f"\n✓ Saved metadata: {METADATA_10S_PATH} with {len(df_meta)} rows.")
print(df_meta.head(5))

In [ ]:
# Cell 6: Frozen WavLM-Base-Plus Feature Extraction (GPU Accelerated)
import gc
from transformers import WavLMModel

MODEL_ID = "microsoft/wavlm-base-plus"
print(f"Loading foundation model: {MODEL_ID}...")
wavlm_model = WavLMModel.from_pretrained(MODEL_ID)
wavlm_model.to(device)
wavlm_model.eval()
wavlm_model.requires_grad_(False)

BATCH_SIZE_FE = 32 if device.type == "cuda" else 8
use_amp = (device.type == "cuda")

pending_indices = []
for idx, row in df_meta.iterrows():
    feat_path = PROJECT_ROOT / row["feature_path"]
    if not feat_path.exists():
        pending_indices.append(idx)

print(f"Total Windows: {len(df_meta)} | Already Cached: {len(df_meta) - len(pending_indices)} | Pending Extraction: {len(pending_indices)}")

if pending_indices:
    for i in tqdm(range(0, len(pending_indices), BATCH_SIZE_FE), desc="Caching WavLM Features"):
        batch_slice = pending_indices[i : i + BATCH_SIZE_FE]
        batch_tensors = []
        batch_dest_paths = []

        for b_idx in batch_slice:
            row = df_meta.iloc[b_idx]
            w_path = PROJECT_ROOT / row["window_path"]
            f_path = PROJECT_ROOT / row["feature_path"]
            arr, _ = sf.read(str(w_path))
            batch_tensors.append(torch.from_numpy(arr).float())
            batch_dest_paths.append(f_path)

        batch_audio = torch.stack(batch_tensors, dim=0).to(device)

        with torch.no_grad():
            if use_amp:
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    feats = wavlm_model(batch_audio).last_hidden_state
            else:
                feats = wavlm_model(batch_audio).last_hidden_state

        feats_np = feats.detach().cpu().to(torch.float16).numpy()

        for feat_arr, f_path in zip(feats_np, batch_dest_paths):
            f_path.parent.mkdir(parents=True, exist_ok=True)
            np.save(str(f_path), feat_arr)

print("✓ All WavLM feature tensors cached successfully.")

# CRITICAL: Delete WavLM model and purge CUDA cache to reclaim ~12 GB VRAM before training!
del wavlm_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"✓ Reclaimed GPU VRAM! Active allocation: {torch.cuda.memory_allocated() / 1e6:.1f} MB (Headroom: ~14.5 GB)")

In [ ]:
# Cell 7: Model Architecture & DataLoaders (Optimized Memory Footprint)
from torch.utils.data import Dataset, DataLoader
from ml.model_def.model import ModelConfig, ParkinsonsVoiceClassifier

class AudioFeatureDataset(Dataset):
    def __init__(self, df: pd.DataFrame, root_dir: Path):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        feat_path = self.root_dir / row["feature_path"]
        feat = np.load(str(feat_path)).astype(np.float32)
        label = np.float32(row["label"])
        return torch.from_numpy(feat), torch.tensor([label], dtype=torch.float32)

train_df = df_meta[df_meta["split"] == "train"]
val_df = df_meta[df_meta["split"] == "val"]
test_df = df_meta[df_meta["split"] == "test"]

train_ds = AudioFeatureDataset(train_df, PROJECT_ROOT)
val_ds = AudioFeatureDataset(val_df, PROJECT_ROOT)
test_ds = AudioFeatureDataset(test_df, PROJECT_ROOT)

# BATCH_SIZE = 16 consumes < 1.2 GB VRAM on T4, preventing OutOfMemoryError
BATCH_SIZE = 16 if device.type == "cuda" else 8
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=(device.type == "cuda"))
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

n_neg = (train_df["label"] == 0).sum()
n_pos = (train_df["label"] == 1).sum()
pos_weight_val = float(n_neg / max(n_pos, 1))

# Initialize 10s Classifier Architecture
config_10s_path = PROJECT_ROOT / "ml" / "model_def" / "model_config_10s.json"
model_cfg = ModelConfig.from_json(config_10s_path)
model = ParkinsonsVoiceClassifier(model_cfg)
model.to(device)

param_counts = model.count_parameters()
print("=" * 55)
print(f"  10-Second Classifier Parameters: {param_counts['total']:,} ({param_counts['total'] / 1e6:.2f}M)")
print(f"  Input Frames T={model_cfg.temporal_frames} -> Downsampled Tokens N={model_cfg.num_tokens}")
print(f"  Batch Size: {BATCH_SIZE} (Memory-safe on T4 GPU)")
print("=" * 55)

In [ ]:
# Cell 8: Mixed-Precision Training Loop
import torch.nn as nn
from sklearn.metrics import roc_auc_score

if torch.cuda.is_available():
    torch.cuda.empty_cache()

pos_weight = torch.tensor([pos_weight_val], device=device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
num_epochs = 20
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)
scaler = torch.amp.GradScaler('cuda', enabled=(device.type == "cuda"))

best_val_auc = 0.0
patience = 7
patience_counter = 0
best_model_weights = None

CHECKPOINT_DIR = PROJECT_ROOT / "models" / "artifact"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
BEST_CHECKPOINT_PATH = CHECKPOINT_DIR / "best_model.pt"

print("Starting model training...")
for epoch in range(1, num_epochs + 1):
    model.train()
    train_loss = 0.0
    for feats, targets in train_loader:
        feats = feats.to(device)
        targets = targets.to(device)
        optimizer.zero_grad()

        with torch.amp.autocast('cuda', enabled=(device.type == "cuda"), dtype=torch.float16):
            logits, _ = model(feats)
            loss = criterion(logits, targets)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item() * len(targets)

    scheduler.step()
    train_loss /= len(train_ds)

    # Validation pass
    model.eval()
    val_loss = 0.0
    val_preds = []
    val_targets = []
    with torch.no_grad():
        for feats, targets in val_loader:
            feats = feats.to(device)
            targets = targets.to(device)
            with torch.amp.autocast('cuda', enabled=(device.type == "cuda"), dtype=torch.float16):
                logits, _ = model(feats)
                loss = criterion(logits, targets)
            val_loss += loss.item() * len(targets)
            val_preds.extend(torch.sigmoid(logits).cpu().numpy().flatten())
            val_targets.extend(targets.cpu().numpy().flatten())

    val_loss /= len(val_ds)
    val_auc = roc_auc_score(val_targets, val_preds)

    print(f"Epoch {epoch:02d}/{num_epochs:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val ROC-AUC: {val_auc:.4f}")

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        patience_counter = 0
        best_model_weights = model.state_dict().copy()
        torch.save({"model_state_dict": best_model_weights, "val_auc": val_auc, "epoch": epoch}, str(BEST_CHECKPOINT_PATH))
        print(f"  -> Best model saved! (ROC-AUC: {best_val_auc:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping triggered at epoch {epoch}.")
            break

model.load_state_dict(best_model_weights)
print(f"✓ Training completed. Optimal Validation ROC-AUC: {best_val_auc:.4f}")

In [ ]:
# Cell 9: Test Evaluation & Temperature Scaling Calibration
from sklearn.metrics import roc_curve, f1_score
from scipy.optimize import minimize

model.eval()
test_logits = []
test_probs = []
test_targets = []

with torch.no_grad():
    for feats, targets in test_loader:
        feats = feats.to(device)
        logits, _ = model(feats)
        test_logits.extend(logits.cpu().numpy().flatten())
        test_probs.extend(torch.sigmoid(logits).cpu().numpy().flatten())
        test_targets.extend(targets.numpy().flatten())

test_probs = np.array(test_probs)
test_targets = np.array(test_targets)
test_auc = roc_auc_score(test_targets, test_probs)

fpr, tpr, thresholds = roc_curve(test_targets, test_probs)
j_scores = tpr - fpr
opt_idx = np.argmax(j_scores)
optimal_threshold = float(thresholds[opt_idx])

test_preds = (test_probs >= optimal_threshold).astype(int)
sensitivity = float(tpr[opt_idx])
specificity = float(1 - fpr[opt_idx])
test_f1 = float(f1_score(test_targets, test_preds))

test_df_eval = test_df.copy()
test_df_eval["prob"] = test_probs
subj_agg = test_df_eval.groupby("subject_id").agg({"prob": "mean", "label": "first"})
subj_auc = roc_auc_score(subj_agg["label"], subj_agg["prob"])

print("=" * 60)
print("             TEST SET EVALUATION RESULTS")
print("=" * 60)
print(f"  Test ROC-AUC (Windows):  {test_auc:.4f}")
print(f"  Subject-Level ROC-AUC:   {subj_auc:.4f} across {len(subj_agg)} test subjects")
print(f"  Optimal Threshold:       {optimal_threshold:.4f}")
print(f"  Sensitivity (Recall):    {sensitivity * 100:.2f}%")
print(f"  Specificity:             {specificity * 100:.2f}%")
print(f"  F1-Score:                {test_f1:.4f}")
print("=" * 60)

# Temperature Scaling Calibration on Validation Logits
val_raw_logits = []
val_raw_targets = []
with torch.no_grad():
    for feats, targets in val_loader:
        feats = feats.to(device)
        logits, _ = model(feats)
        val_raw_logits.extend(logits.cpu().numpy().flatten())
        val_raw_targets.extend(targets.numpy().flatten())

val_raw_logits = np.array(val_raw_logits)
val_raw_targets = np.array(val_raw_targets)

def nll_cost(temp_arr):
    t = temp_arr[0]
    scaled_logits = val_raw_logits / max(t, 1e-3)
    p = 1.0 / (1.0 + np.exp(-scaled_logits))
    eps = 1e-7
    p = np.clip(p, eps, 1 - eps)
    return -np.mean(val_raw_targets * np.log(p) + (1 - val_raw_targets) * np.log(1 - p))

res = minimize(nll_cost, x0=[1.0], bounds=[(0.1, 5.0)])
calibrated_temperature = float(res.x[0])
print(f"✓ Calibrated Temperature: T_scale = {calibrated_temperature:.4f}")

In [ ]:
# Cell 10: Production Export & Automatic Browser Download
import time
import shutil

# 1. Export TorchScript Model
model.eval()
model.to("cpu")
dummy_sample = torch.randn(1, model_cfg.temporal_frames, model_cfg.feature_dim)
traced_model = torch.jit.trace(model, dummy_sample)
TORCHSCRIPT_PATH = CHECKPOINT_DIR / "model.pt"
traced_model.save(str(TORCHSCRIPT_PATH))
print(f"✓ Exported TorchScript model to: {TORCHSCRIPT_PATH}")

# 2. Export updated config.json
export_config = {
    "version": "2.0.0",
    "training_run_id": f"run_10s_multitask_{int(time.time())}",
    "classification_threshold": round(optimal_threshold, 4),
    "temperature": round(calibrated_temperature, 4),
    "feature_dim": 768,
    "T": 499,
    "N": 125,
    "downsample_factor": 4,
    "wavlm_checkpoint": "microsoft/wavlm-base-plus",
    "preprocess_config": {
        "target_sr": 16000,
        "trim_top_db": 30,
        "normalize": True,
        "segment_seconds": 10.0,
        "window_stride": 5.0,
        "min_valid_seconds": 8.0,
        "max_pad_fraction": 0.20,
        "pad_mode": "repeat"
    },
    "risk_tiers": {
        "minimal": [0.0, 0.25],
        "low": [0.25, round(optimal_threshold, 4)],
        "moderate": [round(optimal_threshold, 4), 0.75],
        "high": [0.75, 1.0]
    },
    "export_artifacts": {
        "torchscript_model": "model.pt",
        "pytorch_state_dict": "best_model.pt",
        "eval_metrics": "eval_metrics.json"
    }
}

CONFIG_EXPORT_PATH = CHECKPOINT_DIR / "config.json"
with open(CONFIG_EXPORT_PATH, "w") as f:
    json.dump(export_config, f, indent=2)
print(f"✓ Exported config.json to: {CONFIG_EXPORT_PATH}")

# 3. Export eval_metrics.json
eval_metrics = {
    "test_roc_auc": round(test_auc, 4),
    "test_sensitivity": round(sensitivity, 4),
    "test_specificity": round(specificity, 4),
    "test_f1": round(test_f1, 4),
    "subject_level_roc_auc": round(subj_auc, 4),
    "optimal_threshold": round(optimal_threshold, 4),
    "calibration_temperature": round(calibrated_temperature, 4),
    "test_subjects_count": int(df_meta[df_meta["split"] == "test"]["subject_id"].nunique()),
    "test_windows_count": int(len(test_df))
}
METRICS_EXPORT_PATH = CHECKPOINT_DIR / "eval_metrics.json"
with open(METRICS_EXPORT_PATH, "w") as f:
    json.dump(eval_metrics, f, indent=2)
print(f"✓ Exported eval_metrics.json to: {METRICS_EXPORT_PATH}")

# 4. Create and trigger download of models_artifact_10s.zip
ZIP_PATH = PROJECT_ROOT / "models" / "models_artifact_10s"
shutil.make_archive(str(ZIP_PATH), "zip", str(CHECKPOINT_DIR))
print(f"\n✓ Created production archive: {ZIP_PATH}.zip")

if IN_COLAB:
    print("Triggering automatic browser download of models_artifact_10s.zip...")
    files.download(f"{ZIP_PATH}.zip")
    print("Download initiated! Unzip and copy files into your local models/artifact/ directory.")